# Transformação dos dados

A entrada é `Dados_REG_limpo.tab`. A transformação prepara as 23 atividades para redução de dimensionalidade e duas representações para agrupamento, com e sem `GRAM`. A principal inclui os 506 registros. Uma quarta matriz, ajustada somente aos casos completos, permite testar depois o efeito da escolha de imputação sobre o agrupamento.

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

fonte = Path("Dados_REG_limpo.tab")
pasta_saida = Path("outputs/dados")
dados = pd.read_csv(fonte, sep="\t", encoding="utf-8-sig", dtype="string")

atividades = [f"Atividade {i:02d}" for i in range(1, 24)]
binarias = ["Atividade 01", "Atividade 03", "Atividade 04", "Atividade 16"]
ordinais = ["Atividade 13", *[f"Atividade {i:02d}" for i in range(17, 23)]]
continuas = [coluna for coluna in atividades if coluna not in binarias + ordinais]
com_log = ["Atividade 14", "Atividade 15", "Atividade 23"]
sem_log = [coluna for coluna in continuas if coluna not in com_log]
morfo_simples = ["Borda", "Muco", "Relevo"]
morfo_posicional = ["Cores", "Textura", "Forma", "Brilho"]

print(f"Entrada: {len(dados)} registros e {len(dados.columns)} colunas.")

Entrada: 506 registros e 33 colunas.


## Representação e diagnóstico dos ensaios

As atividades binárias textuais são mapeadas para `Negativo = 0` e `Positivo = 1`. Nas atividades 13 e 17 a 22, usamos a ordem crescente dos níveis 0 a 3. O enunciado não fornece uma legenda para esses níveis, então a ordem e a distância unitária entre níveis adjacentes são hipóteses da representação.

A assimetria é calculada nos valores observados antes da imputação. A coluna após `log1p` aparece para os ensaios contínuos não negativos. A transformação preserva os zeros e comprime valores altos.

In [2]:
ensaios = dados[atividades].copy()
ensaios[binarias] = ensaios[binarias].replace({"Negativo": "0", "Positivo": "1"})
ensaios = ensaios.apply(pd.to_numeric, errors="raise").astype("float64")

valores_continuos = ensaios[continuas]
nao_negativas = valores_continuos.min().ge(0)
diagnostico = pd.DataFrame({
    "Mínimo": valores_continuos.min(),
    "Mediana": valores_continuos.median(),
    "P95": valores_continuos.quantile(0.95),
    "Máximo": valores_continuos.max(),
    "Assimetria original": valores_continuos.skew(),
    "Assimetria após log1p": np.log1p(valores_continuos.loc[:, nao_negativas]).skew(),
})
print(diagnostico.round(3).fillna("—").to_string())

              Mínimo  Mediana      P95   Máximo  Assimetria original Assimetria após log1p
Atividade 02    0.00    0.000    3.330    5.830                0.961                  0.57
Atividade 05    0.00    1.000    1.500    3.600                0.428                -0.035
Atividade 06   -0.06    0.010    0.568    0.810                2.139                     —
Atividade 07   -0.08    0.100    0.360    0.810                2.237                     —
Atividade 08   -0.19    0.010    0.080    0.160               -0.049                     —
Atividade 09    0.00    0.750    1.180    1.510               -0.232                -0.525
Atividade 10    0.00    0.420    0.780    1.080                0.290                -0.056
Atividade 11    0.00    0.240    0.550    0.740                0.774                 0.507
Atividade 12   -0.08    0.175    0.400    0.700                0.963                     —
Atividade 14    0.00    0.000    4.290   15.500                2.882                   1.1

/var/folders/br/99x3j11923v_bc0cvb7ddn6r0000gn/T/ipykernel_9690/4150077567.py:15: Pandas4Warning: 'str' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  print(diagnostico.round(3).fillna("—").to_string())


Aplicamos `log1p` às atividades 14, 15 e 23. Entre as contínuas não negativas, elas são as três com maior assimetria à direita. Nas atividades 14 e 15, a assimetria cai de aproximadamente 2,88 e 2,80 para 1,10 e 1,06. Na atividade 23, cai de 3,14 para 0,15. A concentração de zeros nas atividades 14 e 15 continua visível após o log. As atividades 06, 07, 08 e 12 têm valores negativos e seguem para padronização na escala original.

## Imputação e escala

Preenchemos ausências em atividades contínuas com a **mediana dos valores observados na escala original**, antes do log quando aplicável. Nas binárias e ordinais, usamos a **moda** para manter um nível válido. Nas matrizes principais, só as atividades 20, 21 e 23 precisam ser imputadas.

Padronizamos as atividades contínuas e ordinais pela média e pelo desvio padrão após os tratamentos. As binárias ficam em 0/1 para limitar a influência de respostas pouco frequentes na distância euclidiana. A diferença entre 0 e 1 de uma binária permanece igual a um.

In [3]:
def transformar_atividades(tabela):
    if tabela.isna().all().any():
        vazias = tabela.columns[tabela.isna().all()].tolist()
        raise ValueError(f"Atividades sem valores observados para ajustar: {vazias}")

    pre = ColumnTransformer([
        ("binarias", SimpleImputer(strategy="most_frequent"), binarias),
        ("ordinais", make_pipeline(SimpleImputer(strategy="most_frequent"), StandardScaler()), ordinais),
        ("sem_log", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), sem_log),
        ("com_log", make_pipeline(
            SimpleImputer(strategy="median"),
            FunctionTransformer(np.log1p, feature_names_out="one-to-one"),
            StandardScaler(),
        ), com_log),
    ], verbose_feature_names_out=False)
    valores = pre.fit_transform(tabela)
    matriz = pd.DataFrame(valores, columns=pre.get_feature_names_out(), index=tabela.index)
    return matriz.loc[:, atividades], pre

matriz_atividades, pre_atividades = transformar_atividades(ensaios)

linhas_imputadas = []
for bloco, transformador, colunas in pre_atividades.transformers_:
    if bloco == "binarias":
        imputador = transformador
    else:
        imputador = transformador.named_steps["simpleimputer"]
    for coluna, valor_imputado in zip(colunas, imputador.statistics_):
        for indice in ensaios.index[ensaios[coluna].isna()]:
            linhas_imputadas.append((indice + 1, dados.at[indice, "n°"], coluna, valor_imputado))

auditoria_imputacao = pd.DataFrame(
    linhas_imputadas, columns=["Posição no arquivo", "n°", "Atividade", "Valor imputado"]
)
resumo_imputacao = (
    auditoria_imputacao
    .groupby(["Atividade", "Valor imputado"], as_index=False)
    .size()
    .rename(columns={"size": "Registros"})
)
print(resumo_imputacao.to_string(index=False))

   Atividade  Valor imputado  Registros
Atividade 20          3.0000         27
Atividade 21          3.0000         17
Atividade 23         12.7145          4


A tabela resume a imputação antes da escala. A variável `auditoria_imputacao` contém o detalhe por registro, com a posição no arquivo, o ID original e o valor aplicado. A posição é usada para rastreio dentro do notebook.

In [4]:
print(auditoria_imputacao.to_string(index=False))

 Posição no arquivo   n°    Atividade  Valor imputado
                 61 1492 Atividade 20          3.0000
                 77 1511 Atividade 20          3.0000
                 88 1522 Atividade 20          3.0000
                138 1576 Atividade 20          3.0000
                189 1627 Atividade 20          3.0000
                194 1633 Atividade 20          3.0000
                224 1664 Atividade 20          3.0000
                228 1668 Atividade 20          3.0000
                270 1711 Atividade 20          3.0000
                276 1718 Atividade 20          3.0000
                281 1723 Atividade 20          3.0000
                285 1727 Atividade 20          3.0000
                344 1795 Atividade 20          3.0000
                345 1796 Atividade 20          3.0000
                346 1797 Atividade 20          3.0000
                357 1808 Atividade 20          3.0000
                365 1817 Atividade 20          3.0000
                380 1833 Ati

## Morfologia e GRAM

Cada categoria morfológica é codificada por **one-hot**. Nos campos com vários códigos, cada posição de predominância recebe suas próprias colunas. Assim `3 & 6` ativa `Cores_pos1_3` e `Cores_pos2_6`, e `6 & 3` ativa indicadores diferentes.

Uma célula ausente recebe a categoria `ausente`. Uma observação válida sem código em uma posição seguinte recebe `sem_codigo`. A primeira marca falta de observação e a segunda indica que a observação tem menos códigos registrados.

A contagem `Cor` é usada para conferir a consistência dos dados. Nesta base, ela coincide com a quantidade de códigos em `Cores` em todas as linhas e fica fora da matriz para evitar duplicar essa informação. O ID original também fica fora dos atributos. Na versão com `GRAM`, o one-hot inclui `Desconhecido` para os dois registros ambíguos. Esses dois registros ficam fora da comparação futura em que GRAM funciona como referência.

In [5]:
morfologia = dados[morfo_simples].fillna("ausente")
for coluna in morfo_posicional:
    codigos = dados[coluna].str.split(" & ", regex=False)
    quantidade_maxima = codigos.str.len().max()
    quantidade_posicoes = 1 if pd.isna(quantidade_maxima) else int(quantidade_maxima)
    for posicao in range(quantidade_posicoes):
        nome = f"{coluna}_pos{posicao + 1}"
        morfologia[nome] = (
            codigos.str[posicao].fillna("sem_codigo").mask(dados[coluna].isna(), "ausente")
        )

def codificar_categorias(tabela):
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.uint8)
    valores = encoder.fit_transform(tabela)
    matriz = pd.DataFrame(valores, columns=encoder.get_feature_names_out(), index=tabela.index)
    return matriz

contagem_cor = pd.to_numeric(dados["Cor"], errors="raise")
contagem_codigos = dados["Cores"].str.split(" & ", regex=False).str.len()
comparaveis = contagem_cor.notna() & contagem_codigos.notna()
divergencias = contagem_cor.loc[comparaveis].ne(contagem_codigos.loc[comparaveis]).sum()
print(f"Conferência de Cor e Cores: {divergencias} divergências em {comparaveis.sum()} registros comparáveis.")
if divergencias or (contagem_cor.notna() & contagem_codigos.isna()).any():
    print("Revisar a exclusão de Cor: a contagem pode trazer informação adicional nesta versão.")

bloco_morfologia = codificar_categorias(morfologia)
bloco_gram = codificar_categorias(dados[["GRAM"]].fillna("Desconhecido"))

matriz_sem_gram = pd.concat([matriz_atividades, bloco_morfologia], axis=1)
matriz_com_gram = pd.concat([matriz_sem_gram, bloco_gram], axis=1)
print(f"Indicadores de morfologia: {bloco_morfologia.shape[1]}")
print(f"Indicadores de GRAM: {bloco_gram.shape[1]}")

Conferência de Cor e Cores: 0 divergências em 506 registros comparáveis.
Indicadores de morfologia: 50
Indicadores de GRAM: 4


Entram na distância euclidiana os ensaios na escala definida acima e os indicadores morfológicos em 0/1, com peso unitário por coluna. Para medir a influência desses blocos, calculamos a sua participação média na distância quadrática entre pares de registros. A participação reflete a dispersão observada e o conjunto de colunas de cada bloco.

In [6]:
variancia_blocos = pd.Series({
    "Atividades": matriz_atividades.var(ddof=0).sum(),
    "Morfologia": bloco_morfologia.var(ddof=0).sum(),
    "GRAM": bloco_gram.var(ddof=0).sum(),
})
sem_gram = variancia_blocos.drop("GRAM")
participacao = pd.DataFrame({
    "Sem GRAM": sem_gram / sem_gram.sum(),
    "Com GRAM": variancia_blocos / variancia_blocos.sum(),
})
print(participacao.to_string(na_rep="—", float_format=lambda valor: f"{valor:.1%}"))

            Sem GRAM  Com GRAM
Atividades     83.5%     81.4%
GRAM               —      2.4%
Morfologia     16.5%     16.1%


A representação principal dá mais influência ao perfil dos ensaios do que à morfologia. Usaremos essa distribuição de peso como referência inicial e apresentaremos os perfis morfológicos dos grupos na escala original para facilitar a interpretação.

## Alternativa com linhas completas

Selecionamos os registros com dados disponíveis em todos os atributos usados pelo agrupamento sem `GRAM`. Ajustamos novamente a padronização e o one-hot nesse subconjunto, mantendo as mesmas escolhas de log e de representação. Na análise de sensibilidade, a comparação das partições deve usar somente os registros presentes nas duas versões. A redução da amostra e o reajuste da representação fazem parte do efeito dessa escolha.

In [7]:
colunas_completude = atividades + morfo_simples + morfo_posicional
caso_completo = dados[colunas_completude].notna().all(axis=1)
gram_conhecido = dados["GRAM"].isin(["Negativo", "Positivo", "Levedura"])
referencia = dados[["n°", "GRAM"]].assign(
    gram_conhecido=gram_conhecido, caso_completo=caso_completo
)

atividades_completas, _ = transformar_atividades(ensaios.loc[caso_completo])
morfologia_completa = codificar_categorias(morfologia.loc[caso_completo])
matriz_casos_completos = pd.concat([atividades_completas, morfologia_completa], axis=1)

print(f"Casos completos: {caso_completo.sum()} de {len(dados)}")
print(f"Registros com GRAM conhecido: {gram_conhecido.sum()}")

Casos completos: 468 de 506
Registros com GRAM conhecido: 504


## Exportação e alinhamento

A tabela `referencia_registros.tab` segue a ordem da cópia limpa e das três matrizes principais. Sua coluna `caso_completo` permite localizar as linhas correspondentes à matriz de casos completos, que mantém essa mesma ordem relativa. A posição distingue IDs repetidos durante a leitura. Os arquivos das matrizes contêm apenas atributos.

Os parâmetros das matrizes exportadas foram ajustados em todos os registros de cada versão para a análise exploratória. Se mais adiante houver avaliação supervisionada por validação cruzada, os tratamentos que dependem dos dados devem ser ajustados em cada conjunto de treino.

In [8]:
matrizes_principais = {
    "atividades": matriz_atividades,
    "agrupamento_sem_gram": matriz_sem_gram,
    "agrupamento_com_gram": matriz_com_gram,
}
for matriz in matrizes_principais.values():
    assert matriz.index.equals(referencia.index)

assert matriz_casos_completos.index.equals(referencia.index[caso_completo])
tabelas = {
    **matrizes_principais,
    "agrupamento_sem_gram_casos_completos": matriz_casos_completos,
    "referencia_registros": referencia,
}

for nome, matriz in tabelas.items():
    if nome != "referencia_registros":
        assert np.isfinite(matriz.to_numpy()).all(), f"Valor não finito em {nome}"
        assert matriz.columns.is_unique
        assert not {"n°", "Cor", "linha_origem"}.intersection(matriz.columns)

pasta_saida.mkdir(parents=True, exist_ok=True)
with TemporaryDirectory(dir=pasta_saida) as pasta_temporaria:
    temporaria = Path(pasta_temporaria)
    for nome, tabela in tabelas.items():
        arquivo = temporaria / f"{nome}.tab"
        tabela.to_csv(arquivo, sep="\t", encoding="utf-8-sig", index=False)
        tipos = {"n°": "string", "GRAM": "string"} if nome == "referencia_registros" else None
        reaberta = pd.read_csv(arquivo, sep="\t", encoding="utf-8-sig", dtype=tipos)
        pd.testing.assert_frame_equal(reaberta, tabela.reset_index(drop=True), check_dtype=False)
    for arquivo in sorted(temporaria.iterdir()):
        arquivo.replace(pasta_saida / arquivo.name)

dimensoes = pd.DataFrame(
    [(nome, *tabela.shape) for nome, tabela in tabelas.items()],
    columns=["Arquivo (.tab)", "Linhas", "Colunas"],
)
print(dimensoes.to_string(index=False))

                      Arquivo (.tab)  Linhas  Colunas
                          atividades     506       23
                agrupamento_sem_gram     506       73
                agrupamento_com_gram     506       77
agrupamento_sem_gram_casos_completos     468       64
                referencia_registros     506        4
